In [ ]:
from loguru import logger
import functools
import time

def log_execution(func):
    """记录函数执行的装饰器"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.debug(f"开始执行: {func.__name__}")
        start = time.time()

        try:
            result = func(*args, **kwargs)
            duration = (time.time() - start) * 1000
            logger.success(f"执行完成: {func.__name__} | 耗时: {duration:.2f}ms")
            return result
        except Exception as e:
            duration = (time.time() - start) * 1000
            logger.error(f"执行失败: {func.__name__} | 耗时: {duration:.2f}ms | 错误: {e}")
            raise

    return wrapper

# 使用装饰器
@log_execution
def node_a(state, runtime):
    return {"node_a_output": "node_a is running"}

In [ ]:
# config/logging.py
from loguru import logger
import sys
import os
from pathlib import Path

def setup_logging(env: str = "development"):
    """设置日志配置"""

    # 创建日志目录
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)

    # 移除默认 handler
    logger.remove()

    # 根据环境设置日志级别
    level = "DEBUG" if env == "development" else "INFO"

    # 控制台输出 - 彩色美化
    logger.add(
        sys.stdout,
        format=(
            "<green>{time:YYYY-MM-DD HH:mm:ss}</green> | "
            "<level>{level: <8}</level> | "
            "<cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> | "
            "<level>{message}</level>"
        ),
        colorize=True,
        level=level,
        backtrace=True,
        diagnose=env == "development"
    )

    # 文件输出 - JSON 结构化
    logger.add(
        log_dir / "app_{time:YYYY-MM-DD}.json.log",
        format="{message}",
        serialize=True,
        level="INFO",
        rotation="100 MB",
        retention="30 days",
        compression="gz"
    )

    # 文件输出 - 详细文本
    logger.add(
        log_dir / "app_{time:YYYY-MM-DD}.log",
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message}",
        level=level,
        rotation="100 MB",
        retention="30 days",
        compression="zip"
    )

    # 错误日志单独存储
    logger.add(
        log_dir / "errors_{time:YYYY-MM-DD}.log",
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message}",
        level="ERROR",
        rotation="50 MB",
        retention="90 days",
        backtrace=True,
        diagnose=True
    )

    # 添加默认的上下文信息
    logger.configure(
        extra={
            "app": "langgraph",
            "environment": env,
            "hostname": os.uname().nodename if hasattr(os, 'uname') else "unknown"
        }
    )

    logger.info(f"日志系统初始化完成 | 环境: {env} | 日志级别: {level}")
    return logger

# 在应用入口调用
logger = setup_logging(env="development")

In [ ]:
from loguru import logger
import sys
from datetime import datetime

logger.remove()

# 自定义格式函数
def custom_format(record):
    # 根据日志级别改变颜色
    colors = {
        "DEBUG": "<cyan>",
        "INFO": "<green>",
        "SUCCESS": "<bright-green>",
        "WARNING": "<yellow>",
        "ERROR": "<red>",
        "CRITICAL": "<red><bright>"
    }

    level_color = colors.get(record["level"].name, "")
    reset = "</>" if level_color else ""

    return (
        f"{level_color}[{record['time']:HH:mm:ss.SSS}]</> | "
        f"{level_color}{record['level'].name: <8}</> | "
        f"{record['name']}.{record['function']}:{record['line']} | "
        f"{record['message']}\n"
    )

logger.add(sys.stdout, format=custom_format, colorize=True, level="DEBUG")

# 添加额外的信息到日志上下文
ctx = logger.bind(request_id="12345", user_id="user_001")
ctx.info("处理请求")

In [1]:
from loguru import logger
import sys
from datetime import datetime

logger.remove()

# 定义颜色映射
COLORS = {
    "DEBUG": "cyan",
    "INFO": "green",
    "SUCCESS": "bright_green",
    "WARNING": "yellow",
    "ERROR": "red",
    "CRITICAL": "bright_red"
}

def rich_style_format(record):
    level = record["level"].name
    color = COLORS.get(level, "white")

    # 创建类似 rprint 的格式
    return (
        f"<{color}>[{record['time']:HH:mm:ss}]</{color}> "
        f"<{color}>│</{color}> "
        f"<{color}>{record['level'].name: <8}</{color}> "
        f"<{color}>│</{color}> "
        f"{record['message']}\n"
    )

logger.add(sys.stdout, format=rich_style_format, colorize=True, level="DEBUG")

# 测试
logger.info("信息消息", extra={"key": "value"})
logger.debug("调试消息")
logger.success("成功消息")
logger.warning("警告消息")
logger.error("错误消息")

[11:07:10] │ INFO     │ 信息消息
[11:07:10] │ DEBUG    │ 调试消息
[11:07:10] │ WARNING  │ 警告消息
[11:07:10] │ ERROR    │ 错误消息


--- Logging error in Loguru Handler #1 ---
Record was: {'elapsed': datetime.timedelta(microseconds=8980), 'exception': None, 'extra': {}, 'file': (name='310133893.py', path='C:\\Users\\mike\\AppData\\Local\\Temp\\ipykernel_25356\\310133893.py'), 'function': '<module>', 'level': (name='SUCCESS', no=25, icon='✅'), 'line': 35, 'message': '成功消息', 'module': '310133893', 'name': '__main__', 'process': (id=25356, name='MainProcess'), 'thread': (id=42896, name='MainThread'), 'time': datetime(2026, 9, 5, 11, 7, 10, 110211, tzinfo=datetime.timezone(datetime.timedelta(seconds=28800), '中国标准时间'))}
Traceback (most recent call last):
  File "C:\Users\mike\AppData\Roaming\Python\Python313\site-packages\loguru\_handler.py", line 164, in emit
    _, precomputed_format = self._memoize_dynamic_format(dynamic_format, ansi_level)
                            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mike\AppData\Roaming\Python\Python313\site-packages\loguru\_handler.py", line 

In [ ]:
from loguru import logger
import sys
from rich.console import Console
from rich.table import Table
from rich.pretty import pprint

console = Console()

logger.remove()

# 基础配置
logger.add(
    sys.stdout,
    format=(
        "<green>[{time:HH:mm:ss}]</green> "
        "<level>{level.icon} {level: <7}</level> "
        "<cyan>│</cyan> "
        "<white>{message}</white>"
    ),
    colorize=True,
    level="DEBUG"
)

def rich_log(level: str, message: str, data: dict = None, **kwargs):
    """类似 rprint 的结构化日志"""

    # 将小写级别转换为大写
    level_upper = level.upper()

    if data:
        table = Table(title=message, show_header=False, box=None)
        for key, value in data.items():
            if isinstance(value, dict):
                # 将字典转换为字符串
                value_str = pprint(value, indent_guides=True, expand_all=True)
                # 由于 pprint 直接打印，我们需要转换方式
                table.add_row(f"[cyan]{key}[/cyan]", f"[white]{str(value)}[/white]")
            else:
                table.add_row(f"[cyan]{key}[/cyan]", f"[white]{value}[/white]")

        from io import StringIO
        import sys as sys_io
        old_stdout = sys_io.stdout
        sys_io.stdout = StringIO()
        console.print(table)
        table_str = sys_io.stdout.getvalue()
        sys_io.stdout = old_stdout

        # 使用 loguru 打印 - 使用大写级别
        logger.log(level_upper, f"{message}\n{table_str}")
    else:
        logger.log(level_upper, message)

# 测试
rich_log("INFO", "🚀 系统启动", {
    "version": "1.0.0",
    "environment": "development",
    "config": {
        "debug": True,
        "max_workers": 4
    }
})

rich_log("SUCCESS", "✅ 节点执行成功", {
    "node": "parallel_node_a_1",
    "status": "completed",
    "duration": "150ms"
})

rich_log("ERROR", "❌ 节点执行失败", {
    "node": "parallel_node_a_2",
    "error": "Connection timeout",
    "retry": 3
})